In [36]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def parse_human_readable(value):
    if not value:
        return None
    val = value.replace(',', '')
    last_char = val[-1]
    val_no_last_char = val[:-1]
    match(last_char):
        case 'K':
            return float(val_no_last_char) * 1000
        case 'M':
            return float(val_no_last_char) * 1_000_000
        case 'B':
            return float(val_no_last_char) * 1_000_000_000
        case 'T':
            return float(val_no_last_char) * 1_000_000_000_000
        case _:
            return float(val)


def crypto_volumes():
    response = requests.get(URL, headers=HEADERS)
    print(response)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", attrs={"class": "yf-19xszjc"})
    #print(table)
    rows = table.find('tbody').find_all('tr')
    #print(rows)
    data = []
    for row in rows:
        #print(row)
        print('---------------------')
        cells = row.find_all('td')

        symbol = cells[0].text.strip()
        name = cells[1].text.strip()
        price = cells[4].text.strip()
        market_cap = cells[6].text.strip()
        volume = cells[8].text.strip()
        record = {
            'symbol': symbol,
            'name': name,
            'price': parse_human_readable(price),
            'market_cap': parse_human_readable(market_cap),
            'volume': parse_human_readable(volume)
        }
        print(record)
        data.append(record)
    df = pd.DataFrame(data)
    #df = df.set_index('symbol')
    return df

def save_crypto_volumes(df):
    print(df)
    df.to_csv('data/crypto_volumes.csv', index=False)
    df.to_excel('data/crypto_volumes.xlsx', index=False)
    df.to_json('data/crypto_volumes.json', orient='records', indent=4)
    #df.to_json('data/crypto_volumes.json', index=False)

In [38]:
import plotly.express as px

def plot_volume(df):
    #df_sorted = df.
    fig = px.bar(df, x='symbol', y='volume', title='Crypto Volumes - Yahoo Finance')
    fig.show()

In [39]:
df = crypto_volumes()
save_crypto_volumes(df)
plot_volume(df)

<Response [200]>
---------------------
{'symbol': 'USDT-USD', 'name': 'Tether USDt USD', 'price': 0.0, 'market_cap': 176820000000.0, 'volume': 55675000000.0}
---------------------
{'symbol': 'BTC-USD', 'name': 'Bitcoin USD', 'price': -1691.67, 'market_cap': 1489000000000.0, 'volume': 25264000000.0}
---------------------
{'symbol': 'ETH-USD', 'name': 'Ethereum USD', 'price': -130.15, 'market_cap': 256692000000.0, 'volume': 13998000000.0}
---------------------
{'symbol': 'USDC-USD', 'name': 'USDC USD', 'price': -0.0, 'market_cap': 70243000000.0, 'volume': 9812000000.0}
---------------------
{'symbol': 'SOL-USD', 'name': 'Solana USD', 'price': -2.96, 'market_cap': 50970000000.0, 'volume': 2719000000.0}
---------------------
{'symbol': 'XRP-USD', 'name': 'XRP USD', 'price': -0.07, 'market_cap': 69059000000.0, 'volume': 2649000000.0}
---------------------
{'symbol': 'BNB-USD', 'name': 'BNB USD', 'price': -17.26, 'market_cap': 89460000000.0, 'volume': 1529000000.0}
---------------------
{'sy